In [5]:
from py_module.metadata.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from sqlalchemy import text

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

In [4]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

In [8]:
table_schema = 'public'
table_name = 'fct_meteo'
rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )

In [11]:
it_schema = conn.execute(text(rendered_schema))
columns = list()
for i in it_schema:
    c = {
        i[0]:i[1]
    }
    columns.append(c)

In [15]:

rendered_sources = schema_source.render(
    schema_name = table_schema,
    table_name = table_name,
    cols = columns,
    database = params['database'],
    staging_dataset = 'staging',
    istance_name = params['db'],
    bucket_name = 'postgres__d-meteo-db',
    version = 'v1'
    )

In [ ]:
rendered_cdc = cdc_extraction.render(
                                columns = columns,
                                schema_name = table_schema, 
                                table_name = table_name, 
                                delta = str(delta), 
                                delta_column = delta_column, 
                                delta_timestamp = delta_timestamp, 
                                where_conditions = where_conditions
                            )